# 真实资产与数字孪生

输入 cabinet2 的实拍 RGB-D 与配套 USD，输出米制相机、增强深度、可动外观及与参考模型的配准结果，供物理资产模块使用。

**流程**：输入 → 相机与尺度 → 深度 → 几何与运动 → 配准与校核 → 导出。

实拍只激发下层抽屉；上层独立运动和完整结构来自 USD。尺度与配准使用模型参考，物性保持先验来源。选择正确的解释器后，从第一格全部运行。

## 1. 输入与环境

核对实际解释器和所选方法，读取实拍与 USD。尺寸或命名冲突以显式选择的 USD 为准，并保留在报告中。

In [ ]:
import io
import json
import os
import sys
import zipfile
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("未找到 PhysReal-X 项目根目录。")
    PROJECT_ROOT = PROJECT_ROOT.parent
for path in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from examples._common import (
    NotebookRun,
    operation_receipt,
    output_artifact,
    show_image_strip,
    show_metrics,
    show_rgbd,
)

from physrealx import ArtifactRef, MethodCapability, PhysRealX
from physrealx.integrations.registration import ReferenceRegistrationInput
from physrealx.integrations.rgbd import MetricRGBDInput
from physrealx.integrations.usd_reference import evaluate_reference_kinematics

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
os.environ.setdefault("MAX_JOBS", "3")
RECONSTRUCTION_METHOD = "artgs"
SOURCE = PROJECT_ROOT / "examples/real_assets_and_digital_twin/input/ArtModeling/cabinet2"
OUTPUT_DIR = PROJECT_ROOT / "examples/real_assets_and_digital_twin/output"
run = NotebookRun(
    OUTPUT_DIR / "digital-twin-report.json",
    ("environment", "input", "cameras", "depth", "reconstruction", "registration", "export"),
    {"executable": sys.executable, "python": sys.version, "prefix": sys.prefix},
    evidence_scope="real_capture_with_supplied_usd_reference",
)
with run.stage("environment") as stage:
    show_metrics({"Python": sys.version.split()[0], "解释器": sys.executable})
    if Path(sys.prefix).name != "PhysReal-X":
        raise RuntimeError("请选择现有 PhysReal-X Conda 解释器，并核对其完整路径。")
    import matplotlib.pyplot as plt
    import numpy as np
    from PIL import Image

    figure = plt.figure()
    plt.close(figure)
    px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
    for capability, method in (
        (MethodCapability.CAMERA_RECOVER, "colmap"),
        (MethodCapability.CAMERA_ALIGN_METRIC, "usd_depth_scale"),
        (MethodCapability.DEPTH_ENHANCE, "rgb_guided_fill"),
        (MethodCapability.GEOMETRY_RECONSTRUCT, RECONSTRUCTION_METHOD),
        (MethodCapability.CAMERA_ALIGN_METRIC, "articulated_surface_icp"),
    ):
        status = px.methods.inspect(capability, method)
        if not status.ready:
            raise RuntimeError(f"{method}: {status.detail}")
    stage["versions"] = {
        name: version(name)
        for name in ("numpy", "torch", "matplotlib", "matplotlib-inline", "pycolmap", "usd-core")
    }

In [ ]:
with run.stage("input") as stage:
    inspection = px.assets.inspect_source(SOURCE, importer="artmodeling")
    stage["inspection"] = inspection
    if not inspection.ingestible:
        raise RuntimeError("ArtModeling 数据包未通过结构预检。")
    observations = px.assets.ingest(
        SOURCE,
        resource_id="cabinet2.observations",
        version="1.4.0",
        display_name="Cabinet 2 captures and reverse-engineered USD",
        importer="artmodeling",
        config={
            "depth_unit": "native_u16_unscaled",
            "coordinate_frame": "camera_unregistered",
            "include_candidates": True,
            "view_alignment": {
                "can.start.001.rgb": {
                    "mode": "fiducial_assisted",
                    "fiducial": {"family": "aruco_6x6_50", "layout_id": "cabinet2.table.markers"},
                },
                "can.end.001.rgb": {
                    "mode": "fiducial_assisted",
                    "fiducial": {"family": "aruco_6x6_50", "layout_id": "cabinet2.table.markers"},
                },
                "video.001.rgb": {"mode": "natural_features"},
            },
        },
    )
    reference_model = px.assets.import_reference(observations, usd_path="sim/usd/Cabinet_3.usd")
    reference = json.loads(px.artifacts.get_bytes(reference_model))
    stage.update(
        observations=observations,
        reference_model=reference_model,
        metadata_conflicts=reference["metadata_conflicts"],
    )
    show_metrics(
        {
            "采集流": len(inspection.streams),
            "USD 部件": len(reference["links"]),
            "活动关节": len(reference["joints"]),
            "来源冲突 (已记录)": len(reference["metadata_conflicts"]),
        }
    )

## 2. 相机与米制尺度

恢复两组静态采集的相机，用共享标记连接坐标系，再以 USD 高度确定米制尺度。此步解决长度单位；物体相对于 USD 的位置与朝向在第 5 节求解。

In [ ]:
with run.stage("cameras") as stage:
    camera_operation = px.assets.recover_cameras(
        observations,
        method="colmap",
        config={
            "execution_device": "cpu",
            "matching": "sequential",
            "max_frames_per_stream": 32,
            "num_threads": 1,
            "random_seed": 0,
        },
    )
    camera_rig = output_artifact(camera_operation, "camera_rig_calibration")
    stage["recovery"] = operation_receipt(camera_operation)
    observation_artifact = ArtifactRef(
        artifact_id=observations.artifact_id,
        size_bytes=len(px.artifacts.get_bytes(observations.artifact_id)),
        media_type="application/vnd.physrealx.observation-set+json",
    )
    scale_input = px.artifacts.put_json(
        {
            "schema": "physrealx.model-scale-input/1",
            "observations": observation_artifact.model_dump(mode="json"),
            "reference_model": reference_model.model_dump(mode="json"),
        }
    )
    metric_operation = px.assets.align_cameras_metric(
        camera_rig, scale_input, method="usd_depth_scale", config={}
    )
    metric_camera = output_artifact(metric_operation, "camera_rig_calibration")
    stage["metric"] = operation_receipt(metric_operation)
    show_metrics(
        {
            "注册视角": int(camera_operation.metrics["registered_image_count"]),
            "重投影误差 (px)": camera_operation.metrics["mean_reprojection_error_px"],
            "高度留出相对误差": metric_operation.metrics["max_height_holdout_relative_error"],
        }
    )

## 3. 深度准备与增强

绑定 RGB、深度与相机，每状态最多选 16 帧、像素步长为 2。补全有邻域支持的小孔，以遮挡有效像素的方式检查误差；大孔洞与断层继续保留无效标记。

In [ ]:
with run.stage("depth") as stage:
    calibrated_rgbd = px.assets.prepare_rgbd(
        observations, metric_camera, stride=2, max_frames_per_stream=16
    )
    stage["calibrated_rgbd"] = calibrated_rgbd
    depth_operation = px.assets.enhance_depth(
        calibrated_rgbd, method="rgb_guided_fill", config={"max_rmse_m": 0.02}
    )
    enhanced_depth = output_artifact(depth_operation, "enhanced_depth")
    stage["operation"] = operation_receipt(depth_operation)
    rgbd_input = MetricRGBDInput(
        camera_rig=metric_camera, enhanced_depth=enhanced_depth, reference_model=reference_model
    )
    geometry_input = px.artifacts.put_json(rgbd_input)
    stage["reconstruction_input"] = geometry_input
    show_metrics({"自监督 RMSE (m)": depth_operation.metrics["holdout_rmse_m"]})
    with zipfile.ZipFile(io.BytesIO(px.artifacts.get_bytes(enhanced_depth))) as archive:
        first = json.loads(archive.read("manifest.json"))["frames"][0]
        depth_image = np.load(io.BytesIO(archive.read(first["depth"]["path"])), allow_pickle=False)
        rgb_image = np.load(
            io.BytesIO(px.artifacts.get_bytes(ArtifactRef.model_validate(first["rgb"]))),
            allow_pickle=False,
        )
    show_rgbd(rgb_image, depth_image, OUTPUT_DIR / "rgbd-preview.png")

## 4. 几何与运动重建

使用上一步的统一米制输入，按 `method` 选择实现，恢复可观测几何、运动组和连续外观。当前两状态仅下层抽屉运动，预期为两个运动组；下方展示留出视角的五个中间状态。

In [ ]:
with run.stage("reconstruction") as stage:
    reconstruction_operation = px.assets.reconstruct_geometry(
        geometry_input,
        method=RECONSTRUCTION_METHOD,
        config={
            "num_slots": 2,
            "state_streams": ["can.start.001.rgb", "can.end.001.rgb"],
            "coarse_iterations": 5000,
            "joint_iterations": 3000,
            "reconstruction_iterations": 10000,
            "training_resolution": 1,
            "evaluation_resolution": 1,
            "seed": 2026,
            "max_stage_seconds": 1800,
        },
    )
    stage["operation"] = operation_receipt(reconstruction_operation)
    reconstruction_report_ref = output_artifact(
        reconstruction_operation, "geometry_reconstruction_report"
    )
    reconstruction_report = json.loads(px.artifacts.get_bytes(reconstruction_report_ref))
    assert reconstruction_report["evaluation_scope"] == "held_out_views"
    assert reconstruction_report["motion_group_count"] == 2
    metrics = reconstruction_report["metrics"]
    assert metrics["validation_fg_psnr"] > 12 and metrics["validation_alpha_iou"] > 0.75
    show_metrics(
        {
            "运动组": reconstruction_report["motion_group_count"],
            "留出前景 PSNR (dB)": metrics["validation_fg_psnr"],
            "留出轮廓 IoU": metrics["validation_alpha_iou"],
        }
    )
    reconstruction_bundle = output_artifact(reconstruction_operation, "visual_geometry_bundle")
    with zipfile.ZipFile(io.BytesIO(px.artifacts.get_bytes(reconstruction_bundle))) as archive:
        previews = {
            f"State {index / 4:.2f}": np.array(
                Image.open(io.BytesIO(archive.read(f"appearance/{index:02d}.png")))
            )
            for index in range(5)
        }
    show_image_strip(previews, OUTPUT_DIR / "reconstruction-preview.png")

## 5. 位姿配准与关节校核

结合支撑平面与重建运动轴，将实拍结果配准到 USD，并用留出视角检查距离误差。随后核对 USD 闭合与展开构型，供物理结构和重建外观共同使用。

In [ ]:
with run.stage("registration") as stage:
    registration_input = px.artifacts.put_json(
        ReferenceRegistrationInput(rgbd=rgbd_input, reconstruction_report=reconstruction_report_ref)
    )
    registration_operation = px.assets.align_cameras_metric(
        metric_camera,
        registration_input,
        method="articulated_surface_icp",
        config={"joint": "gui_drawer_2.joint"},
    )
    reference_registration = output_artifact(registration_operation, "metric_alignment_report")
    registered_cameras = output_artifact(registration_operation, "camera_rig_calibration")
    stage["operation"] = operation_receipt(registration_operation)
    joint_positions_closed = {joint["name"]: joint["lower_limit"] for joint in reference["joints"]}
    joint_positions_open = {joint["name"]: joint["upper_limit"] for joint in reference["joints"]}
    closed_kinematics = evaluate_reference_kinematics(
        reference_model, px.artifacts, joint_positions_closed
    )
    open_kinematics = evaluate_reference_kinematics(
        reference_model, px.artifacts, joint_positions_open
    )
    stage.update(closed=closed_kinematics, open=open_kinematics)
    show_metrics(
        {
            "留出最大中位距离 (mm)": registration_operation.metrics["max_holdout_median_m"] * 1000,
            "留出最大 P90 距离 (mm)": registration_operation.metrics["max_holdout_p90_m"] * 1000,
            "校核关节数": len(reference["joints"]),
        }
    )

## 6. 结果与导出

导出下一模块所需的参考结构、重建外观和配准结果。页面保留关键指标与预览，完整方法配置、来源及质量报告保存在 `output/`；必要阶段失败会停止执行。

In [ ]:
with run.stage("export") as stage:
    exports = {
        "reference-model.json": reference_model,
        "reference-usd.zip": ArtifactRef.model_validate(reference["bundle"]),
        "reference-registration.json": reference_registration,
        "registered-cameras.json": registered_cameras,
        "metric-cameras.json": metric_camera,
        "enhanced-depth.zip": enhanced_depth,
        "reconstruction-input.json": geometry_input,
        "reconstruction.zip": reconstruction_bundle,
        "reconstruction-report.json": reconstruction_report_ref,
        "kinematics-closed.json": closed_kinematics,
        "kinematics-open.json": open_kinematics,
    }
    with TemporaryDirectory(prefix=".export-", dir=OUTPUT_DIR) as directory:
        for filename, artifact in exports.items():
            px.artifacts.materialize(artifact, Path(directory) / filename)
        for filename in exports:
            (Path(directory) / filename).replace(OUTPUT_DIR / filename)
    stage.update(artifacts=exports, output_directory=str(OUTPUT_DIR))
report = run.complete()
show_metrics(
    {
        "执行状态": report["execution_status"],
        "完成阶段": len(report["stages"]),
        "结果目录": OUTPUT_DIR,
    }
)